# trackmater: Single-Particle Tracking Analysis in Python

A Python port of [TrackMateR](https://github.com/quantixed/TrackMateR) for analysing [TrackMate](https://imagej.net/plugins/trackmate/) (ImageJ/Fiji) XML outputs.

This notebook walks through a complete analysis of a single tracking dataset: loading data, recalibration, visualising tracks, computing MSD, jump distance analysis, fractal dimension, track density, and generating a summary report.

In [ ]:
import trackmater as tm
from pathlib import Path
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. Loading Data

`read_trackmate_xml()` reads a TrackMate XML file and returns a `TrackMateData` object containing the track DataFrame and calibration metadata.

In [ ]:
xml_path = Path(tm.__file__).parent / "data" / "ExampleTrackMateData.xml"
data = tm.read_trackmate_xml(xml_path)

In [ ]:
data

In [ ]:
data.calibration

## 2. Recalibrating Data

The example data uses pixel coordinates. If you know the pixel size (here 0.04 µm), use `.correct()` to rescale to physical units. This returns a new `TrackMateData` object (the original is unchanged).

In [ ]:
data = data.correct(xy_scalar=0.04, xy_unit="um")
data.calibration

## 3. Visualising Tracks

In [ ]:
fig, ax = tm.plot_tracks(data)

In [ ]:
fig, ax = tm.plot_displacement_over_time(data)

In [ ]:
fig, ax = tm.plot_cumulative_distance(data)

## 4. Distribution Plots

Histograms summarise per-track or per-step statistics. Each returns `(fig, ax, median_value)`. All plot functions accept an optional `ax` parameter for composing multi-panel figures.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

tm.plot_displacement_hist(data, ax=axes[0, 0])
tm.plot_speed(data, ax=axes[0, 1])
tm.plot_duration_hist(data, ax=axes[1, 0])
tm.plot_intensity_hist(data, ax=axes[1, 1])

fig.tight_layout()

## 5. Mean Squared Displacement (MSD)

MSD quantifies diffusive behaviour. `calculate_msd()` computes time-averaged (default) or ensemble MSD, plus per-track anomalous exponents (alpha) and covariance-based diffusion coefficient estimates (CVE).

In [ ]:
msd = tm.calculate_msd(data.tracks, n=3, short=8)
msd

In [ ]:
units = (data.calibration.spatial_unit, data.calibration.temporal_unit)
fig, ax, dee = tm.plot_msd(msd.summary, units=units)
print(f"Diffusion coefficient D = {dee:.4f} {units[0]}\u00b2/{units[1]}")

In [ ]:
alpha_df = msd.alpha.dropna(subset=["alpha"])
valid = alpha_df[(alpha_df["alpha"] >= -4) & (alpha_df["alpha"] <= 4)]
median_alpha = 2 ** float(valid["alpha"].median())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
tm.plot_alpha(valid, median_alpha=median_alpha, ax=axes[0])
tm.plot_dee(valid, median_dee=float(valid["dee"].median()), ax=axes[1])
fig.tight_layout()

## 6. Jump Distance Analysis

Jump distance analysis decomposes particle displacements into diffusive populations. Fit the empirical CDF to 1, 2, or 3 population models using `fit_jd()`.

In [ ]:
jd = data.calculate_jd(delta_t=1, n_pop=2)
jd

In [ ]:
jd_fit = tm.fit_jd(jd)
print("Fitted coefficients:")
for k, v in jd_fit.coefficients.items():
    print(f"  {k} = {v:.4f}")

In [ ]:
plt.close(jd_fit.fig)

## 7. Fractal Dimension

The fractal dimension (FD) describes track complexity. Values near 1 indicate directed motion; values near 2 indicate random walks. The maximum width measures the spatial extent of each track.

In [ ]:
fd = data.calculate_fd()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
tm.plot_fd(fd.data, ax=axes[0])
tm.plot_width(fd.data, units=units, ax=axes[1])
fig.tight_layout()

## 8. Track Density

Track density estimates how many neighbouring tracks start near each track, correcting for edge effects at image boundaries.

In [ ]:
td = data.calculate_track_density(radius=1.5)
fig, ax, median_density = tm.plot_neighbours(td.data)
print(f"Median track density: {median_density:.2f}")

## 9. Summary Report

`report_dataset()` runs the full analysis pipeline and generates a multi-panel figure in one call.

In [ ]:
fig = data.report(n=3, short=8, delta_t=1, n_pop=2, radius=1.5)

The functional API gives access to both the figure and a `SummaryStats` object with all median values:

In [ ]:
msd_result = tm.calculate_msd(data.tracks, n=3, short=8)
jd_result = tm.calculate_jd(data, delta_t=1, n_pop=2)
td_result = tm.calculate_track_density(data, radius=1.5)
fd_result = tm.calculate_fd(data)

fig, stats = tm.make_summary_report(
    data, msd_result,
    jd_result=jd_result,
    td_result=td_result,
    fd_result=fd_result,
    title="Example Dataset",
)
print(f"Summary statistics:")
print(f"  Alpha:       {stats.alpha:.3f}")
print(f"  Speed:       {stats.speed:.4f} {units[0]}/{units[1]}")
print(f"  Duration:    {stats.duration:.2f} {units[1]}")
print(f"  D (MSD):     {stats.dee:.4f}")
print(f"  FD:          {stats.fd:.3f}")
print(f"  Density:     {stats.neighbours:.2f}")

## 10. Exporting Data

Use `to_csv()` to export track data, optionally filtering by minimum track length or converting back to pixel coordinates.

In [ ]:
export_df = tm.to_csv(data, min_points=10)
print(f"Exported {len(export_df)} rows from {export_df['trace'].nunique()} tracks")
export_df.head()

## 11. Working with the Data Directly

The underlying data is a pandas DataFrame, so you can use standard pandas operations for custom analysis.

In [ ]:
df = data.tracks

# Track lengths (number of points per track)
track_lengths = df.groupby("trace").size()
print(f"Track length: min={track_lengths.min()}, median={track_lengths.median():.0f}, max={track_lengths.max()}")

# Plot a single track
single_track = df[df["trace"] == data.trace_ids[0]]
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(single_track["x"], single_track["y"], "o-", markersize=2)
ax.set_aspect("equal")
ax.set_title(f"Track {data.trace_ids[0]}")
ax.set_xlabel(f"x ({data.calibration.spatial_unit})")
ax.set_ylabel(f"y ({data.calibration.spatial_unit})")

## What's Next

- **Batch comparison**: Use `tm.compare_datasets("Data/")` to analyse multiple conditions at once. Organise your XML files into subdirectories (one per condition) under a `Data/` folder.
- **Custom initial guesses**: Pass `init={"D1": 0.001, "D2": 0.5, "D3": 0.1}` to `calculate_jd()` if the automatic fit does not converge.
- **Different MSD methods**: Try `method="ensemble"` in `calculate_msd()` for ensemble-averaged MSD.
- **Log-scale MSD**: Pass `xlog=True, ylog=True` to `plot_msd()` to visualise anomalous diffusion on log-log axes.